In [2]:
# pip install "sympy<1.13"

In [3]:
import pandas as pd

In [4]:
# Run this cell if your CSV file is in Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Set the path to the directory containing the csv files
csv_location = '/content/drive/MyDrive/ML Project/Dataset/'

In [6]:
# Load the csv files
box_office_df = pd.read_csv(csv_location + "model_2_dataset.csv")

In [7]:
box_office_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1517 entries, 0 to 1516
Data columns (total 34 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rank                  1517 non-null   int64  
 1   title                 1517 non-null   object 
 2   worldwide_revenue     1517 non-null   int64  
 3   domestic_revenue      1517 non-null   int64  
 4   domestic_percent      1517 non-null   float64
 5   foreign_revenue       1517 non-null   int64  
 6   foreign_percent       1517 non-null   float64
 7   year                  1517 non-null   int64  
 8   genres                1517 non-null   object 
 9   is_action             1517 non-null   int64  
 10  is_adventure          1517 non-null   int64  
 11  is_animation          1517 non-null   int64  
 12  is_comedy             1517 non-null   int64  
 13  is_crime              1517 non-null   int64  
 14  is_documentary        1517 non-null   int64  
 15  is_drama             

# **Preprocessing Steps**



## 0.   Feature Selection





In [8]:
TARGET_COLUMN = "is_winner"
drop_columns = [TARGET_COLUMN, 'genres', 'title']

X = box_office_df.drop(columns=drop_columns)
y = box_office_df[TARGET_COLUMN]

## 1. Encoding

In [9]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer

# Categorical encoding
le = LabelEncoder()
y = np.array(le.fit_transform(y))

X['original_language'] = le.fit_transform(X['original_language'])
X['production_countries'] = le.fit_transform(X['production_countries'])
X['director'] = le.fit_transform(X['director'])


## 2. Split Data

In [10]:
from sklearn.model_selection import train_test_split

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

# Split training set into training and validation set
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train, test_size=0.2, random_state=42, stratify=y_train)


## 3. Handle Class Imbalance

In [11]:
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight

# SMOTE (Oversampling)
classes, counts = np.unique(y_train, return_counts=True)
print(f"Split before SMOTE: {classes[0]} -> {counts[0]}, {classes[1]} -> {counts[1]}")
smote = SMOTE(sampling_strategy=2/3, random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

classes, counts = np.unique(y_train, return_counts=True)
print(f"Split after SMOTE: {classes[0]} -> {counts[0]}, {classes[1]} -> {counts[1]}")

# Class weights
class_weights_array = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, class_weights_array))
print(f"Class weights: {class_weights_dict}")

Split before SMOTE: 0 -> 956, 1 -> 14
Split after SMOTE: 0 -> 956, 1 -> 637
Class weights: {np.int64(0): np.float64(0.8331589958158996), np.int64(1): np.float64(1.250392464678179)}


## 4. Scale Features

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train)
validate_scaled = scaler.transform(X_val)
test_scaled = scaler.transform(X_test)

In [13]:
print(f"Training set size: {train_scaled.shape[0]}")
print(f"Validation set size: {validate_scaled.shape[0]}")
print(f"Test set size: {test_scaled.shape[0]}")

Training set size: 1593
Validation set size: 243
Test set size: 304


# **Building the Neural Network**

In [14]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import InputLayer, Dense, Dropout

In [15]:
nIn = train_scaled.shape[1]
nHidden = (2 * nIn) // 3
nOut = 1

model = Sequential([
    InputLayer(shape=(nIn,)),
    Dense(nHidden, activation="relu"),
    Dense(nOut, activation="sigmoid"),
])


In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_scaled, y_train,
    validation_data=(validate_scaled, y_val),
    class_weight=class_weights_dict,
    epochs=20,
    batch_size=128,
    verbose=1
  )

Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5141 - loss: 0.7424 - val_accuracy: 0.7901 - val_loss: 0.4475
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6522 - loss: 0.6103 - val_accuracy: 0.8477 - val_loss: 0.3758
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7778 - loss: 0.5101 - val_accuracy: 0.8683 - val_loss: 0.3236
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8707 - loss: 0.4363 - val_accuracy: 0.9053 - val_loss: 0.2857
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9058 - loss: 0.3770 - val_accuracy: 0.9300 - val_loss: 0.2573
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9278 - loss: 0.3297 - val_accuracy: 0.9547 - val_loss: 0.2340
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9360 - loss: 0.2893 - val_accuracy: 0.9465 - val_loss: 0.2128
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9404 - loss: 0.2561 - val_accuracy: 0.9506 - val_loss